In [1]:
# Import required libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn

from sklearn.ensemble import RandomForestRegressor

# Check environment
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("All required packages imported successfully.")

Python: 3.14.2 (main, Oct 28 2026, 19:03:49) [GCC 13.3.0]
Python executable: /usr/local/python/3.14.2/bin/python
NumPy: 2.5.5
Pandas: 3.1.1
Scikit-learn: 1.9.1
All required packages imported successfully.


In [2]:
# Load river level and monitoring location datasets
df_avg = pd.read_csv("avg_river_levels.csv")
df_min = pd.read_csv("min_river_levels.csv")
df_max = pd.read_csv("max_river_levels.csv")
df_locs = pd.read_csv("monitoring_locations.csv")

# Check dataset dimensions
print("Average river levels:", df_avg.shape)
print("Minimum river levels:", df_min.shape)
print("Maximum river levels:", df_max.shape)
print("Monitoring locations:", df_locs.shape)

Average river levels: (392, 4382)
Minimum river levels: (392, 4382)
Maximum river levels: (392, 4382)
Monitoring locations: (392, 5)


In [3]:
# Define temporal boundaries
train_end = pd.Timestamp("2025-09-01")
test_start = pd.Timestamp("2026-09-01")
test_end = pd.Timestamp("2026-12-01")

# Align datasets by observation ID
df_min = (
    df_min.set_index("Observation_ID")
    .loc[df_avg["Observation_ID"]]
    .reset_index()
)

df_max = (
    df_max.set_index("Observation_ID")
    .loc[df_avg["Observation_ID"]]
    .reset_index()
)

# Identify date columns
level_cols = df_avg.columns[3:]
dates = pd.to_datetime(level_cols, format="%d/%m/%Y")


# ------------------------------------------------------------
# 2.2 Anomaly Removal
# ------------------------------------------------------------

A = df_avg[level_cols].to_numpy(dtype=float)
MN = df_min[level_cols].to_numpy(dtype=float)
MX = df_max[level_cols].to_numpy(dtype=float)

# Identify sensor error codes
error_code = (MX >= 999999) | (MN <= -999999)

# Identify impossible water levels
impossible = (
    (np.abs(A) > 100) |
    (np.abs(MN) > 100) |
    (np.abs(MX) > 100)
)

obvious = error_code | impossible

# Calculate station medians using training data only
train_cols = dates < train_end
glitch = np.zeros(A.shape, dtype=bool)

for i in range(A.shape[0]):
    row = A[i, train_cols]
    ok = ~obvious[i, train_cols] & np.isfinite(row)

    if not ok.any():
        continue

    med = np.median(row[ok])

    glitch[i] = (
        (MN[i] < med - 5) |
        (MX[i] > med + 15)
    ) & ~obvious[i]

# Combine anomaly masks
remove = obvious | glitch

# Replace anomalous observations with missing values
df_avg[level_cols] = df_avg[level_cols].mask(remove)
df_min[level_cols] = df_min[level_cols].mask(remove)
df_max[level_cols] = df_max[level_cols].mask(remove)

print("\n=== Anomaly Removal ===")
print("Sensor error codes:", error_code.sum())
print("Impossible values:", (impossible & ~error_code).sum())
print("Glitches:", glitch.sum())
print("Total removed:", remove.sum())


# ------------------------------------------------------------
# 2.3 Missing Data Handling
# ------------------------------------------------------------

# Calculate daily data coverage
coverage = df_avg[level_cols].notna().mean(axis=0)

# Apply the coverage threshold to training dates only
keep_mask = (coverage.to_numpy() >= 0.1) | (dates >= train_end)
keep_dates = level_cols[keep_mask]

# Create cleaned datasets
df_avg_clean = pd.concat(
    [df_avg.iloc[:, :3], df_avg[keep_dates]], axis=1
)

id_cols = ["County", "Station_Name", "Observation_ID"]

df_min_clean = pd.concat(
    [df_min[id_cols], df_min[keep_dates]], axis=1
)

df_max_clean = pd.concat(
    [df_max[id_cols], df_max[keep_dates]], axis=1
)

print("\n=== Missing Data Handling ===")
print("Original dates:", len(level_cols))
print("Retained dates:", len(keep_dates))
print("Removed dates:", len(level_cols) - len(keep_dates))


# ------------------------------------------------------------
# 2.4 Data Reshaping and Merging
# ------------------------------------------------------------

# Convert wide-format datasets to long format
df_long_avg = df_avg_clean.melt(
    id_vars=id_cols,
    var_name="date",
    value_name="avg"
)

df_long_min = df_min_clean.melt(
    id_vars=id_cols,
    var_name="date",
    value_name="min"
)

df_long_max = df_max_clean.melt(
    id_vars=id_cols,
    var_name="date",
    value_name="max"
)

# Convert date columns to datetime
for data in [df_long_avg, df_long_min, df_long_max]:
    data["date"] = pd.to_datetime(
        data["date"], format="%d/%m/%Y"
    )

# Merge river level datasets using observation ID and date
df = (
    df_long_avg
    .merge(
        df_long_min[["Observation_ID", "date", "min"]],
        on=["Observation_ID", "date"],
        validate="one_to_one"
    )
    .merge(
        df_long_max[["Observation_ID", "date", "max"]],
        on=["Observation_ID", "date"],
        validate="one_to_one"
    )
)

# Merge geographical information
df = df.merge(
    df_locs[["Observation_ID", "Latitude", "Longitude"]],
    on="Observation_ID",
    how="left",
    validate="many_to_one"
)

# Sort observations chronologically
df = df.sort_values(
    ["Observation_ID", "date"]
).reset_index(drop=True)

print("\n=== Data Reshaping and Merging ===")
print("Merged dataset:", df.shape)
print("Number of monitoring records:", df["Observation_ID"].nunique())


# ------------------------------------------------------------
# 2.5 Feature Engineering
# ------------------------------------------------------------

# Seasonal features
df["day_of_year"] = df["date"].dt.dayofyear

df["sin_day"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["cos_day"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

# Calculate station statistics using training data only
station_stats = (
    df.loc[df["date"] < train_end]
    .groupby("Observation_ID")["avg"]
    .agg(
        station_mean="mean",
        station_std="std"
    )
    .reset_index()
)

# Merge station-level statistics
df = df.merge(
    station_stats,
    on="Observation_ID",
    how="left",
    validate="many_to_one"
)

# Create calendar-day lag features
lags = [1, 7, 30, 90, 365, 730]

history = df.set_index(
    ["Observation_ID", "date"]
)["avg"]

for lag in lags:
    lag_index = pd.MultiIndex.from_arrays([
        df["Observation_ID"],
        df["date"] - pd.Timedelta(days=lag)
    ])

    df[f"lag_{lag}"] = history.reindex(lag_index).to_numpy()

# Define model features
feature_cols = [
    "lag_1",
    "lag_7",
    "lag_30",
    "lag_90",
    "lag_365",
    "lag_730",
    "Latitude",
    "Longitude",
    "station_mean",
    "station_std",
    "sin_day",
    "cos_day"
]

target_col = "avg"

print("\n=== Feature Engineering ===")
print("Number of features:", len(feature_cols))

# Check missing values in model variables
print("\nMissing values:")
print(df[feature_cols + [target_col]].isna().sum())


# ------------------------------------------------------------
# 2.6 Train / Validation / Test Split
# ------------------------------------------------------------

# Retain complete observations for model evaluation
df_model = df.dropna(
    subset=feature_cols + [target_col]
).copy()

# Chronological data splitting
train = df_model[
    df_model["date"] < train_end
].copy()

valid = df_model[
    (df_model["date"] >= train_end) &
    (df_model["date"] < test_start)
].copy()

test = df_model[
    (df_model["date"] >= test_start) &
    (df_model["date"] < test_end)
].copy()

# Display dataset information
print("\n=== Dataset Split ===")

for name, data in [
    ("Training", train),
    ("Validation", valid),
    ("Test", test)
]:
    print(f"\n{name} set")
    print("Shape:", data.shape)
    print("Start date:", data["date"].min())
    print("End date:", data["date"].max())
    print("Monitoring records:", data["Observation_ID"].nunique())


# ------------------------------------------------------------
# 2.7 Feature and Target Preparation
# ------------------------------------------------------------

# Training set
X_train = train[feature_cols]
y_train = train[target_col]

# Validation set
X_val = valid[feature_cols]
y_val = valid[target_col]

# Test set
X_test = test[feature_cols]
y_test = test[target_col]

# Check final dimensions
print("\n=== Final Model Inputs ===")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# Validate model inputs
for X, y in [
    (X_train, y_train),
    (X_val, y_val),
    (X_test, y_test)
]:
    assert len(X) == len(y)
    assert len(X) > 0
    assert np.isfinite(X.to_numpy()).all()
    assert np.isfinite(y.to_numpy()).all()

print("\nData preparation completed successfully.")

/tmp/ipykernel_14363/571651407.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
/tmp/ipykernel_14363/571651407.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()



=== Anomaly Removal ===
Sensor error codes: 76
Impossible values: 4
Glitches: 280
Total removed: 360

=== Missing Data Handling ===
Original dates: 4379
Retained dates: 3294
Removed dates: 1085

=== Data Reshaping and Merging ===
Merged dataset: (1291248, 9)
Number of monitoring records: 392


/tmp/ipykernel_14363/571651407.py:176: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df["day_of_year"] = df["date"].dt.dayofyear



=== Feature Engineering ===
Number of features: 12

Missing values:
lag_1           109381
lag_7           118498
lag_30          139004
lag_90          182388
lag_365         308169
lag_730         436520
Latitude             0
Longitude            0
station_mean         0
station_std          0
sin_day              0
cos_day              0
avg             107453
dtype: int64

=== Dataset Split ===

Training set
Shape: (566081, 20)
Start date: 2019-03-31 00:00:00
End date: 2025-08-31 00:00:00
Monitoring records: 389

Validation set
Shape: (123889, 20)
Start date: 2025-09-01 00:00:00
End date: 2026-08-31 00:00:00
Monitoring records: 370

Test set
Shape: (13595, 20)
Start date: 2026-09-01 00:00:00
End date: 2026-10-08 00:00:00
Monitoring records: 366

=== Final Model Inputs ===
X_train: (566081, 12)
y_train: (566081,)
X_val: (123889, 12)
y_val: (123889,)
X_test: (13595, 12)
y_test: (13595,)

Data preparation completed successfully.


In [6]:
# Define a recursive multi-step forecasting function

def recursive_forecast(model, df, forecast_start, forecast_end,
                       feature_cols, lags):

    forecast_start = pd.Timestamp(forecast_start)
    forecast_end = pd.Timestamp(forecast_end)

    # Generate all forecast dates
    forecast_dates = pd.date_range(
        start=forecast_start,
        end=forecast_end,
        freq="D"
    )

    # Use only observations before the forecast start
    historical = df.loc[
        df["date"] < forecast_start,
        ["Observation_ID", "date", "avg"]
    ].copy()

    assert historical["date"].max() < forecast_start

    # Extract static station information
    stations = (
        df[["Observation_ID", "Latitude", "Longitude"]]
        .drop_duplicates(subset="Observation_ID")
        .set_index("Observation_ID")
    )

    station_ids = stations.index

    # Calculate station statistics using historical data only
    station_stats = (
        historical.groupby("Observation_ID")["avg"]
        .agg(station_mean="mean", station_std="std")
    )

    stations = stations.join(station_stats)

    # Convert historical observations into a date-by-station matrix
    history = (
        historical.pivot(
            index="date",
            columns="Observation_ID",
            values="avg"
        )
        .reindex(columns=station_ids)
        .sort_index()
    )

    # Create a complete daily calendar
    all_dates = pd.date_range(
        start=history.index.min(),
        end=forecast_end,
        freq="D"
    )

    history = history.reindex(all_dates)

    # Ensure that no future observations are present
    assert history.loc[forecast_dates].isna().all().all()

    predictions = []
    missing_count = 0

    # Predict each day sequentially
    for current_date in forecast_dates:

        X_day = stations.copy()

        # Construct lag features from historical or predicted values
        for lag in lags:

            lag_date = current_date - pd.Timedelta(days=lag)

            if lag_date in history.index:
                X_day[f"lag_{lag}"] = history.loc[lag_date]
            else:
                X_day[f"lag_{lag}"] = np.nan

        # Generate seasonal features
        day_of_year = current_date.dayofyear

        X_day["sin_day"] = np.sin(
            2 * np.pi * day_of_year / 365.25
        )

        X_day["cos_day"] = np.cos(
            2 * np.pi * day_of_year / 365.25
        )

        # Replace missing lag values with historical station means
        for lag in lags:

            col = f"lag_{lag}"

            missing_count += int(X_day[col].isna().sum())

            X_day[col] = X_day[col].fillna(
                X_day["station_mean"]
            )

        # Keep the same feature order as the training data
        X_day = X_day[feature_cols]

        # Predict water levels for all stations on the current date
        y_pred = model.predict(X_day)

        # Store predictions as inputs for subsequent dates
        history.loc[current_date, station_ids] = y_pred

        # Store daily forecast results
        daily_predictions = pd.DataFrame({
            "Observation_ID": station_ids,
            "date": current_date,
            "predicted_avg": y_pred
        })

        predictions.append(daily_predictions)

    # Combine all daily predictions
    forecast = pd.concat(
        predictions,
        ignore_index=True
    )

    # Validate forecast completeness
    expected_predictions = len(station_ids) * len(forecast_dates)

    assert len(forecast) == expected_predictions
    assert not forecast.duplicated(
        ["Observation_ID", "date"]
    ).any()
    assert np.isfinite(forecast["predicted_avg"]).all()

    print("\n=== Recursive Forecast Summary ===")
    print("Historical data ends:", historical["date"].max())
    print("Forecast start:", forecast_start.date())
    print("Forecast end:", forecast_end.date())
    print("Forecast days:", len(forecast_dates))
    print("Monitoring records:", len(station_ids))
    print("Total predictions:", len(forecast))
    print("Missing lag values encountered:", missing_count)

    return forecast


print("Recursive forecasting function defined successfully.")

Recursive forecasting function defined successfully.


In [9]:
# Evaluate the baseline random forest using a fixed forecast origin

val_forecast_start = "2025-09-01"
val_forecast_end = "2025-11-30"

start_time = time.time()

rf_val_forecast = recursive_forecast(
    model=rf_baseline,
    df=df,
    forecast_start=val_forecast_start,
    forecast_end=val_forecast_end,
    feature_cols=feature_cols,
    lags=lags
)

forecast_time = time.time() - start_time

# Extract actual observations for evaluation only
val_actual = df.loc[
    (df["date"] >= pd.Timestamp(val_forecast_start)) &
    (df["date"] <= pd.Timestamp(val_forecast_end)),
    ["Observation_ID", "date", "avg"]
].copy()

# Match predictions with actual observations
rf_val_evaluation = rf_val_forecast.merge(
    val_actual,
    on=["Observation_ID", "date"],
    how="left",
    validate="one_to_one"
)

# Exclude unavailable actual observations from evaluation
rf_val_evaluation = rf_val_evaluation.dropna(
    subset=["avg", "predicted_avg"]
)

# Calculate evaluation metrics
recursive_mae = mean_absolute_error(
    rf_val_evaluation["avg"],
    rf_val_evaluation["predicted_avg"]
)

recursive_rmse = np.sqrt(
    mean_squared_error(
        rf_val_evaluation["avg"],
        rf_val_evaluation["predicted_avg"]
    )
)

recursive_r2 = r2_score(
    rf_val_evaluation["avg"],
    rf_val_evaluation["predicted_avg"]
)

print("\n=== Multi-step Baseline Evaluation ===")
print(f"Forecasting time: {forecast_time:.2f} seconds")
print("Total predictions:", len(rf_val_forecast))
print("Evaluated predictions:", len(rf_val_evaluation))

print(
    f"Evaluation coverage: "
    f"{len(rf_val_evaluation) / len(rf_val_forecast):.2%}"
)

print(f"Validation MAE: {recursive_mae:.4f}")
print(f"Validation RMSE: {recursive_rmse:.4f}")
print(f"Validation R²: {recursive_r2:.4f}")

/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2025-08-31 00:00:00
Forecast start: 2025-09-01
Forecast end: 2025-11-30
Forecast days: 91
Monitoring records: 392
Total predictions: 35672
Missing lag values encountered: 6032

=== Multi-step Baseline Evaluation ===
Forecasting time: 3.24 seconds
Total predictions: 35672
Evaluated predictions: 33985
Evaluation coverage: 95.27%
Validation MAE: 0.2087
Validation RMSE: 0.2851
Validation R²: 0.9711


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear


In [10]:
# Define candidate hyperparameter combinations
param_configs = [
    ("RF1", 50, 10, 5, 1.0),
    ("RF2", 50, 20, 5, 1.0),
    ("RF3", 50, 15, 10, 1.0),
    ("RF4", 50, 15, 5, 0.7),
    ("RF5", 100, 15, 5, 1.0)
]

# Store multi-step baseline results
results = [{
    "Model": "Baseline",
    "n_estimators": 50,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "MAE": recursive_mae,
    "RMSE": recursive_rmse,
    "R2": recursive_r2,
    "Training_Time": training_time
}]

# Initialise the best model
best_rf = rf_baseline
best_rmse = recursive_rmse
best_model_name = "Baseline"

# Train and evaluate candidate models
for name, trees, depth, leaf, features in param_configs:

    print(f"\nTraining {name}...", flush=True)

    model = RandomForestRegressor(
        n_estimators=trees,
        max_depth=depth,
        min_samples_leaf=leaf,
        max_features=features,
        min_samples_split=2,
        random_state=42,
        n_jobs=2
    )

    # Train the model
    start_time = time.time()

    model.fit(X_train, y_train)

    elapsed_time = time.time() - start_time

    # Generate recursive multi-step forecasts
    forecast = recursive_forecast(
        model=model,
        df=df,
        forecast_start=val_forecast_start,
        forecast_end=val_forecast_end,
        feature_cols=feature_cols,
        lags=lags
    )

    # Match predictions with actual observations
    evaluation = forecast.merge(
        val_actual,
        on=["Observation_ID", "date"],
        how="left",
        validate="one_to_one"
    ).dropna(subset=["avg", "predicted_avg"])

    # Ensure consistent evaluation coverage
    assert len(evaluation) == len(rf_val_evaluation)

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        evaluation["avg"],
        evaluation["predicted_avg"]
    )

    rmse = np.sqrt(mean_squared_error(
        evaluation["avg"],
        evaluation["predicted_avg"]
    ))

    r2 = r2_score(
        evaluation["avg"],
        evaluation["predicted_avg"]
    )

    # Store results
    results.append({
        "Model": name,
        "n_estimators": trees,
        "max_depth": depth,
        "min_samples_leaf": leaf,
        "max_features": features,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time": elapsed_time
    })

    # Select the best model using multi-step RMSE
    if rmse < best_rmse:
        best_rmse = rmse
        best_rf = model
        best_model_name = name

    print(f"\n{name} Results")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"Training time: {elapsed_time:.2f} seconds")

# Create the model comparison table
tuning_results = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\n=== Hyperparameter Tuning Results ===")

display(tuning_results.round(4))

print("\nBest model:", best_model_name)
print(f"Best validation RMSE: {best_rmse:.4f}")
print("Best parameters:", best_rf.get_params())


Training RF1...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2025-08-31 00:00:00
Forecast start: 2025-09-01
Forecast end: 2025-11-30
Forecast days: 91
Monitoring records: 392
Total predictions: 35672
Missing lag values encountered: 6032

RF1 Results
MAE: 0.2106
RMSE: 0.2891
R²: 0.9702
Training time: 97.63 seconds

Training RF2...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2025-08-31 00:00:00
Forecast start: 2025-09-01
Forecast end: 2025-11-30
Forecast days: 91
Monitoring records: 392
Total predictions: 35672
Missing lag values encountered: 6032

RF2 Results
MAE: 0.2100
RMSE: 0.2861
R²: 0.9709
Training time: 171.05 seconds

Training RF3...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur

... (4 outputs left out)


=== Recursive Forecast Summary ===
Historical data ends: 2025-08-31 00:00:00
Forecast start: 2025-09-01
Forecast end: 2025-11-30
Forecast days: 91
Monitoring records: 392
Total predictions: 35672
Missing lag values encountered: 6032

RF5 Results
MAE: 0.2095
RMSE: 0.2848
R²: 0.9711
Training time: 270.28 seconds

=== Hyperparameter Tuning Results ===


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear


,Model,n_estimators,max_depth,min_samples_leaf,max_features,MAE,RMSE,R2,Training_Time
0,RF5,100,15,5,1.0,0.2095,0.2848,0.9711,270.2818
1,Baseline,50,15,5,1.0,0.2087,0.2851,0.9711,141.4314
2,RF2,50,20,5,1.0,0.2100,0.2861,0.9709,171.0490
3,RF3,50,15,10,1.0,0.2091,0.2867,0.9708,135.0955
4,RF4,50,15,5,0.7,0.2139,0.2891,0.9702,91.9779
5,RF1,50,10,5,1.0,0.2106,0.2891,0.9702,97.6269



Best model: RF5
Best validation RMSE: 0.2848
Best parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 15, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 5, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': 2, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


Six Random Forest configurations were evaluated using recursive multi-step forecasting over September–November 2025. RF5 achieved the lowest validation RMSE of 0.2848, with an R² of 0.9711. Increasing the number of trees from 50 to 100 resulted in only a marginal improvement in RMSE while nearly doubling the training time. Nevertheless, RF5 was selected as the best-performing Random Forest configuration based on validation RMSE.

In [11]:
# Define additional validation windows
validation_windows = [
    ("Fold 2", "2025-12-01", "2026-02-28"),
    ("Fold 3", "2026-03-01", "2026-05-31"),
    ("Fold 4", "2026-06-01", "2026-08-31")
]

# Define the two candidate configurations
candidate_models = {
    "Baseline": 50,
    "RF5": 100
}

# Reuse the existing Fold 1 results
temporal_results = []

for name in candidate_models:

    row = tuning_results.loc[
        tuning_results["Model"] == name
    ].iloc[0]

    temporal_results.append({
        "Fold": "Fold 1",
        "Model": name,
        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],
        "Evaluated": len(rf_val_evaluation),
        "Training_Time": row["Training_Time"]
    })

# Perform expanding-window validation
for fold, forecast_start, forecast_end in validation_windows:

    print(f"\n{'=' * 50}")
    print(f"{fold}: {forecast_start} to {forecast_end}")
    print(f"{'=' * 50}")

    cutoff = pd.Timestamp(forecast_start)

    # Select historical training data
    fold_train = df.loc[df["date"] < cutoff].copy()

    # Recalculate station statistics using historical data only
    fold_stats = (
        fold_train.groupby("Observation_ID")["avg"]
        .agg(
            station_mean="mean",
            station_std="std"
        )
        .reset_index()
    )

    fold_train = (
        fold_train
        .drop(columns=["station_mean", "station_std"])
        .merge(
            fold_stats,
            on="Observation_ID",
            how="left",
            validate="many_to_one"
        )
    )

    # Remove incomplete training samples
    fold_train = fold_train.dropna(
        subset=feature_cols + [target_col]
    )

    X_fold = fold_train[feature_cols]
    y_fold = fold_train[target_col]

    assert fold_train["date"].max() < cutoff

    print("Training samples:", len(X_fold))
    print("Training ends:", fold_train["date"].max())

    # Select actual observations for evaluation only
    fold_actual = df.loc[
        (df["date"] >= cutoff) &
        (df["date"] <= pd.Timestamp(forecast_end)),
        ["Observation_ID", "date", "avg"]
    ].copy()

    # Compare Baseline and RF5
    for name, trees in candidate_models.items():

        print(f"\nTraining {name}...", flush=True)

        model = RandomForestRegressor(
            n_estimators=trees,
            max_depth=15,
            min_samples_leaf=5,
            min_samples_split=2,
            max_features=1.0,
            random_state=42,
            n_jobs=2
        )

        start_time = time.time()

        model.fit(X_fold, y_fold)

        elapsed_time = time.time() - start_time

        # Generate recursive forecasts
        forecast = recursive_forecast(
            model=model,
            df=df,
            forecast_start=forecast_start,
            forecast_end=forecast_end,
            feature_cols=feature_cols,
            lags=lags
        )

        # Evaluate against available observations
        evaluation = forecast.merge(
            fold_actual,
            on=["Observation_ID", "date"],
            how="left",
            validate="one_to_one"
        ).dropna(subset=["avg", "predicted_avg"])

        mae = mean_absolute_error(
            evaluation["avg"],
            evaluation["predicted_avg"]
        )

        rmse = np.sqrt(mean_squared_error(
            evaluation["avg"],
            evaluation["predicted_avg"]
        ))

        r2 = r2_score(
            evaluation["avg"],
            evaluation["predicted_avg"]
        )

        temporal_results.append({
            "Fold": fold,
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Evaluated": len(evaluation),
            "Training_Time": elapsed_time
        })

        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"R²: {r2:.4f}")
        print(f"Training time: {elapsed_time:.2f} seconds")

        del model

# Create detailed validation results
temporal_results_df = pd.DataFrame(temporal_results)

print("\n=== Temporal Validation Results ===")
display(temporal_results_df.round(4))

# Compare average performance across all four folds
temporal_summary = (
    temporal_results_df.groupby("Model")
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Std_RMSE=("RMSE", "std"),
        Mean_R2=("R2", "mean"),
        Mean_Training_Time=("Training_Time", "mean")
    )
    .sort_values("Mean_RMSE")
)

print("\n=== Overall Model Comparison ===")
display(temporal_summary.round(4))


Fold 2: 2025-12-01 to 2026-02-28
Training samples: 598649
Training ends: 2025-11-30 00:00:00

Training Baseline...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2025-11-30 00:00:00
Forecast start: 2025-12-01
Forecast end: 2026-02-28
Forecast days: 90
Monitoring records: 392
Total predictions: 35280
Missing lag values encountered: 6748
MAE: 0.2526
RMSE: 0.3439
R²: 0.9576
Training time: 143.98 seconds

Training RF5...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2025-11-30 00:00:00
Forecast start: 2025-12-01
Forecast end: 2026-02-28
Forecast days: 90
Monitoring records: 392
Total predictions: 35280
Missing lag values encountered: 6748
MAE: 0.2540
RMSE: 0.3491
R²: 0.9563
Training time: 286.15 seconds

Fold 3: 2026-03-01 to 2026-05-31
Training samples: 630073
Training ends: 2026-02-26 00:00:00

Training Baseline...


/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur

... (7 outputs left out)

/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear


,Fold,Model,MAE,RMSE,R2,Evaluated,Training_Time
0,Fold 1,Baseline,0.2087,0.2851,0.9711,33985,141.4314
1,Fold 1,RF5,0.2095,0.2848,0.9711,33985,270.2818
2,Fold 2,Baseline,0.2526,0.3439,0.9576,33594,143.9794
3,Fold 2,RF5,0.2540,0.3491,0.9563,33594,286.1472
4,Fold 3,Baseline,0.2081,0.2806,0.9716,34319,152.0066
5,Fold 3,RF5,0.2057,0.2756,0.9726,34319,303.3017
6,Fold 4,Baseline,0.1816,0.2542,0.9759,34149,159.4358
7,Fold 4,RF5,0.1828,0.2545,0.9758,34149,319.4851



=== Overall Model Comparison ===


,Mean_MAE,Mean_RMSE,Std_RMSE,Mean_R2,Mean_Training_Time
Model,,,,,
Baseline,0.2127,0.291,0.0378,0.969,149.2133
RF5,0.2130,0.291,0.0408,0.969,294.8039


The two Random Forest configurations were compared using expanding-window validation across four seasonal periods. Both models achieved a mean RMSE of approximately 0.2910 and a mean R² of 0.9690.

However, the model with 50 trees achieved a slightly lower mean MAE and lower variability in RMSE across the validation windows, while requiring approximately 49.4% less training time. Therefore, the 50-tree configuration was selected as the final Random Forest model, providing a better balance between predictive performance and computational efficiency.

In [12]:
# Define the final forecast period
final_forecast_start = "2026-09-01"
final_forecast_end = "2026-11-30"

final_cutoff = pd.Timestamp(final_forecast_start)

# Select all available historical data
final_train = df.loc[
    df["date"] < final_cutoff
].copy()

# Recalculate station statistics
final_station_stats = (
    final_train.groupby("Observation_ID")["avg"]
    .agg(
        station_mean="mean",
        station_std="std"
    )
    .reset_index()
)

# Replace the previous station statistics
final_train = (
    final_train
    .drop(columns=["station_mean", "station_std"])
    .merge(
        final_station_stats,
        on="Observation_ID",
        how="left",
        validate="many_to_one"
    )
)

# Remove incomplete training samples
final_train = final_train.dropna(
    subset=feature_cols + [target_col]
)

# Prepare final training inputs
X_final = final_train[feature_cols]
y_final = final_train[target_col]

# Check training data
assert final_train["date"].max() < final_cutoff
assert np.isfinite(X_final.to_numpy()).all()
assert np.isfinite(y_final.to_numpy()).all()

print("=== Final Training Dataset ===")
print("Training samples:", len(X_final))
print("Number of features:", len(feature_cols))
print("Training starts:", final_train["date"].min())
print("Training ends:", final_train["date"].max())
print(
    "Monitoring records:",
    final_train["Observation_ID"].nunique()
)

print(
    f"Training sample increase: "
    f"{(len(X_final) / len(X_train) - 1):.2%}"
)

# Define the final Random Forest model
rf_final = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    min_samples_split=2,
    max_features=1.0,
    random_state=42,
    n_jobs=2
)

# Train the final model
start_time = time.time()

rf_final.fit(X_final, y_final)

final_training_time = time.time() - start_time

print("\n=== Final Random Forest Model ===")
print(f"Training time: {final_training_time:.2f} seconds")
print("Number of trees:", len(rf_final.estimators_))
print("Final model trained successfully.")

=== Final Training Dataset ===
Training samples: 689970
Number of features: 12
Training starts: 2019-03-31 00:00:00
Training ends: 2026-08-31 00:00:00
Monitoring records: 389
Training sample increase: 21.89%

=== Final Random Forest Model ===
Training time: 168.87 seconds
Number of trees: 50
Final model trained successfully.


In [13]:
# Generate final recursive forecasts
start_time = time.time()

rf_final_forecast = recursive_forecast(
    model=rf_final,
    df=df,
    forecast_start=final_forecast_start,
    forecast_end=final_forecast_end,
    feature_cols=feature_cols,
    lags=lags
)

final_forecast_time = time.time() - start_time

# Identify monitoring records absent from the training set
unseen_ids = set(df["Observation_ID"].unique()) - set(
    final_train["Observation_ID"].unique()
)

# Check forecast completeness
assert len(rf_final_forecast) == 392 * 91
assert rf_final_forecast["Observation_ID"].nunique() == 392

print("\n=== Final Forecast Results ===")
print(f"Forecasting time: {final_forecast_time:.2f} seconds")
print("Forecast starts:", rf_final_forecast["date"].min())
print("Forecast ends:", rf_final_forecast["date"].max())
print("Total predictions:", len(rf_final_forecast))
print("Monitoring records absent from training:", len(unseen_ids))

display(rf_final_forecast.head())

/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = current_date.dayofyear
/tmp/ipykernel_14363/2076162052.py:83: Pandas4Warning: Timestamp.dayofyear is deprecated and will be removed in a future version. Use Timestamp.day_of_year instead.
  day_of_year = cur


=== Recursive Forecast Summary ===
Historical data ends: 2026-08-31 00:00:00
Forecast start: 2026-09-01
Forecast end: 2026-11-30
Forecast days: 91
Monitoring records: 392
Total predictions: 35672
Missing lag values encountered: 6063

=== Final Forecast Results ===
Forecasting time: 3.29 seconds
Forecast starts: 2026-09-01 00:00:00
Forecast ends: 2026-11-30 00:00:00
Total predictions: 35672
Monitoring records absent from training: 3


,Observation_ID,date,predicted_avg
0,aberdeenshire__bogie-redcraig__obs1,2026-09-01,0.134095
1,aberdeenshire__dee-grampian-mar-lodge__obs1,2026-09-01,0.655466
2,aberdeenshire__dee-grampian-park__obs1,2026-09-01,0.771368
3,aberdeenshire__dee-grampian-polhollick__obs1,2026-09-01,0.387629
4,aberdeenshire__dee-grampian-woodend__obs1,2026-09-01,0.407792


In [14]:
# Select available actual observations
final_actual = df.loc[
    (df["date"] >= pd.Timestamp(final_forecast_start)) &
    (df["date"] <= pd.Timestamp(final_forecast_end)),
    ["Observation_ID", "date", "avg"]
].copy()

# Match predictions with actual observations
rf_final_evaluation = rf_final_forecast.merge(
    final_actual,
    on=["Observation_ID", "date"],
    how="left",
    validate="one_to_one"
)

# Retain records with available actual observations
rf_final_evaluation = rf_final_evaluation.dropna(
    subset=["avg", "predicted_avg"]
).copy()

# Calculate prediction errors
rf_final_evaluation["error"] = (
    rf_final_evaluation["predicted_avg"] -
    rf_final_evaluation["avg"]
)

rf_final_evaluation["absolute_error"] = (
    rf_final_evaluation["error"].abs()
)

# Calculate final evaluation metrics
final_mae = mean_absolute_error(
    rf_final_evaluation["avg"],
    rf_final_evaluation["predicted_avg"]
)

final_rmse = np.sqrt(mean_squared_error(
    rf_final_evaluation["avg"],
    rf_final_evaluation["predicted_avg"]
))

final_r2 = r2_score(
    rf_final_evaluation["avg"],
    rf_final_evaluation["predicted_avg"]
)

# Display final evaluation results
print("=== Final Random Forest Evaluation ===")

print("Total predictions:", len(rf_final_forecast))
print("Evaluated predictions:", len(rf_final_evaluation))

print(
    f"Evaluation coverage: "
    f"{len(rf_final_evaluation) / len(rf_final_forecast):.2%}"
)

print("Actual observations start:",
      rf_final_evaluation["date"].min())

print("Actual observations end:",
      rf_final_evaluation["date"].max())

print(f"Test MAE: {final_mae:.4f}")
print(f"Test RMSE: {final_rmse:.4f}")
print(f"Test R²: {final_r2:.4f}")

=== Final Random Forest Evaluation ===
Total predictions: 35672
Evaluated predictions: 14098
Evaluation coverage: 39.52%
Actual observations start: 2026-09-01 00:00:00
Actual observations end: 2026-10-08 00:00:00
Test MAE: 0.1582
Test RMSE: 0.2200
Test R²: 0.9023
